In [86]:
import pandas as pd


# load models
import joblib

pt = joblib.load("../models/power_transformer.pkl")

# scaler = joblib.load("../models/scaler.pkl")

# FEATURES = joblib.load("../models/features.pkl")

In [87]:
test = pd.read_csv("../data/test.csv")

test

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,projected_advance_m,dist_accel_m_per_h2,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month
0,10662602,1,0.000000,1,2.452217,0.000000,0.00000,0.000000,1.239017,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,3,7
1,13353600,1,0.000000,1,131.669588,0.000000,0.00000,0.000000,4.887862,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,22,0,8
2,13942327,1,0.000000,1,6.723104,0.000000,0.00000,0.000000,2.044216,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,6,7
3,16112781,1,0.000000,1,285.416736,0.000000,0.00000,0.000000,5.657448,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,7
4,17132808,7,3.459331,0,61.098604,12.516633,0.20486,3.618224,4.128724,2.603921,...,13.54413,-22.687575,0.044572,0.158550,0.158550,-24.414806,3.920562,23,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,94627327,2,4.614923,0,24.438326,0.000000,0.00000,0.000000,3.236257,0.000000,...,0.00000,0.000000,0.000000,0.763809,0.763809,0.000000,0.000000,22,0,6
91,96570675,1,0.000000,1,155.843418,0.000000,0.00000,0.000000,5.055248,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,21,2,7
92,97225766,1,0.000000,1,87.761553,0.000000,0.00000,0.000000,4.485954,0.000000,...,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,7
93,98446281,2,4.335964,0,11.391108,0.000000,0.00000,0.000000,2.516979,0.000000,...,0.00000,0.000000,0.000000,-0.305683,0.305683,0.000000,-0.000000,17,0,9


In [88]:
# distance speed
test["distance_speed"] = test["dist_min_ci_0_5h"] / (test["area_first_ha"] + 1)

# direction alignment
test["direction_alignment"] = test["spread_bearing_cos"] * test["alignment_cos"]


# direction alignment
test["direction_alignment"] = test["spread_bearing_cos"] * test["alignment_cos"]


test["fire_pressure"] = test["area_first_ha"] / (test["dist_min_ci_0_5h"] + 1)


NameError: name 'kj' is not defined

In [90]:
# ------------------------------------------------
# IDENTIFIER
# ------------------------------------------------

ID_COL = ["event_id"]

# ------------------------------------------------
# TARGET
# ------------------------------------------------

TARGET_TIME = ["time_to_hit_hours"]
TARGET_EVENT = ["event"]

# ------------------------------------------------
# CATEGORICAL / BINARY
# ------------------------------------------------

categorical = [
    "low_temporal_resolution_0_5h"
]

# ------------------------------------------------
# CATEGORICAL COUNT
# ------------------------------------------------

categorical_count = [
    "num_perimeters_0_5h"
]

# ------------------------------------------------
# TIME FEATURES
# ------------------------------------------------

time_features = [
    "dt_first_last_0_5h",
    "event_start_hour",
    "event_start_dayofweek",
    "event_start_month"
]

# ------------------------------------------------
# CONTINUOUS FEATURES
# ------------------------------------------------

continuous = [

    # fire size / growth
    "area_first_ha",
    "area_growth_abs_0_5h",
    "area_growth_rel_0_5h",
    "area_growth_rate_ha_per_h",
    "log1p_area_first",
    "log1p_growth",
    "log_area_ratio_0_5h",
    "relative_growth_0_5h",

    # radial growth
    "radial_growth_m",
    "radial_growth_rate_m_per_h",

    # centroid movement
    "centroid_displacement_m",
    "centroid_speed_m_per_h",

    # spread direction
    "spread_bearing_deg",
    "spread_bearing_sin",
    "spread_bearing_cos",

    # distance to evacuation zone
    "dist_min_ci_0_5h",
    "dist_std_ci_0_5h",
    "dist_change_ci_0_5h",
    "dist_slope_ci_0_5h",

    # closing speed
    "closing_speed_m_per_h",
    "closing_speed_abs_m_per_h",
    "projected_advance_m",

    # distance dynamics
    "dist_accel_m_per_h2",
    "dist_fit_r2_0_5h",

    # direction alignment
    "alignment_cos",
    "alignment_abs",

    # vector movement
    "cross_track_component",
    "along_track_speed",

    # engineered features
    "distance_speed",
    "direction_alignment",
    "fire_pressure"
]

# ------------------------------------------------
# ALL FEATURES
# ------------------------------------------------

ALL_FEATURES = (
    continuous
    + categorical
    + categorical_count
    + time_features
)

print("Continuous:", len(continuous))
print("Categorical:", len(categorical))
print("Categorical Count:", len(categorical_count))
print("Time Features:", len(time_features))
print("Total Features:", len(ALL_FEATURES))

Continuous: 31
Categorical: 1
Categorical Count: 1
Time Features: 4
Total Features: 37


In [91]:
# Drop Those Features

test = test.drop(columns=["along_track_speed"])

In [95]:
# Check Remaining Features

print("Remaining features:", test.columns)

Remaining features: Index(['event_id', 'num_perimeters_0_5h', 'dt_first_last_0_5h',
       'low_temporal_resolution_0_5h', 'area_first_ha', 'area_growth_abs_0_5h',
       'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h', 'log1p_area_first',
       'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h',
       'radial_growth_m', 'radial_growth_rate_m_per_h',
       'centroid_displacement_m', 'centroid_speed_m_per_h',
       'spread_bearing_deg', 'spread_bearing_sin', 'spread_bearing_cos',
       'dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h',
       'dist_slope_ci_0_5h', 'closing_speed_m_per_h',
       'closing_speed_abs_m_per_h', 'projected_advance_m',
       'dist_accel_m_per_h2', 'dist_fit_r2_0_5h', 'alignment_cos',
       'alignment_abs', 'cross_track_component', 'event_start_hour',
       'event_start_dayofweek', 'event_start_month', 'distance_speed',
       'direction_alignment', 'fire_pressure'],
      dtype='object')


In [97]:
drop_vif = [
    "area_growth_rel_0_5h",
    "relative_growth_0_5h",
    "projected_advance_m",
    "dist_change_ci_0_5h"
]

test = test.drop(columns=drop_vif)

In [98]:
drop_growth = [
    "area_growth_rate_ha_per_h",
    "area_growth_abs_0_5h",
    "radial_growth_rate_m_per_h",
    "radial_growth_m"
]
drop_motion = [
    "centroid_displacement_m"
]

drop_distance = [
    "dist_std_ci_0_5h",
    "dist_slope_ci_0_5h"
]

drop_direction = [
    "spread_bearing_cos",
    "spread_bearing_sin"
]

drop_alignment = [
    "alignment_cos",
    "direction_alignment"
]


drop_cols = (
    drop_growth
    + drop_motion
    + drop_distance
    + drop_direction
    + drop_alignment
)

test = test.drop(columns=drop_cols)

In [99]:
drop_cols = [
"closing_speed_m_per_h",
"closing_speed_abs_m_per_h"
]

test = test.drop(columns=drop_cols)

In [100]:
# Drop Event ID
test = test.drop(columns=["event_id"])

In [101]:
test.shape

(95, 19)

In [102]:
test["dist_accel_m_per_h2"] = pt.transform(test[["dist_accel_m_per_h2"]])

In [ ]:
dfhb

In [ ]:
test[FEATURES] = scaler.transform(test[FEATURES])
HORIZONS = [12,24,48,72]
surv = model.predict_survival_function(
    test[FEATURES],
    times=HORIZONS
)

probs = 1 - surv.T.values
probs = np.maximum.accumulate(probs, axis=1)
submission = pd.DataFrame({

    "event_id": event_ids,

    "prob_12h": probs[:,0],
    "prob_24h": probs[:,1],
    "prob_48h": probs[:,2],
    "prob_72h": probs[:,3]

})


In [ ]:
submission.to_csv("../data/submission.csv", index=False)

print("Submission file saved")

Submission file saved


In [ ]:
features = [
"dist_min_ci_0_5h",
"area_first_ha",
"dt_first_last_0_5h",
"num_perimeters_0_5h"
]

X_test = test[features]

event_prob = lgb_cls.predict_proba(X_test)[:,1]
time_pred = lgb_reg.predict(X_test)

In [ ]:
# predict time
time_pred = lgb_reg.predict(X_test)

# convert to probabilities
prob_12h = (time_pred <= 12).astype(float)
prob_24h = (time_pred <= 24).astype(float)
prob_48h = (time_pred <= 48).astype(float)
prob_72h = (time_pred <= 72).astype(float)

# create submission
submission = pd.DataFrame({
    "event_id": event_ids,
    "prob_12h": prob_12h,
    "prob_24h": prob_24h,
    "prob_48h": prob_48h,
    "prob_72h": prob_72h
})

# save
submission.to_csv("../submission/submission.csv", index=False)

submission.head()

Dataset Shape: (221, 22)

========== SURVIVAL CV RESULTS ==========
C-index: 0.90725
Brier@12: 0.07571
Brier@24: 0.08155
Brier@48: 0.07894
Brier@72: 0.10846

Weighted Brier: 0.08858
Hybrid Score: 0.91017
